# 7. Modelo vencedor e escoragem da OOT

Comparação dos três modelos, escolher o vencedor, conferir uma única vez na OOT interna, explicar o vencedor e escorar a OOT cega

1. **M1** (Modelo Base): LightGBM ajustado no `03_modelo_base`, só com as colunas fornecidas
- **M2** (Feature Engineering): M1 + família de CEP (`04_feature_engineering`).
- **M3** (Macro): M2 + colunas macro (`06_modelo_macro`)

- Os três modelos usam os mesmos hiperparâmetros do LightGBM (`reports/params_m1.json`)
- A escolha usa só o que já foi medido sem a OOT interna (teste por cliente e validação temporal). A OOT interna (202309–202310) é usada uma única vez, depois da escolha, para confirmar
- O modelo entregue é retreinado com as 10 safras (202301–202310), com os mesmos hiperparâmetros, e escora a OOT cega (`data/raw/base_oot.csv`). O score é a probabilidade de adimplência (score maior = risco menor), numa coluna `score` no fim do arquivo original.

## 7.1 Imports e leitura

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from equifax.avaliacao import (grafico_calibracao, grafico_distribuicao_score, psi_score_por_safra,
                               simular_politica, tabela_calibracao, tabela_decis)
from equifax.config import MODELS, OOT_PATH, RANDOM_SEED, REPORTS, SUBMISSION, TABLES
from equifax.conjuntos import carregar_conjunto
from equifax.dados import carregar_desenvolvimento, janelas
from equifax.escoragem import (colunas_brutas_do_desenvolvimento, ensaiar_com_desenvolvimento,
                               escorar_arquivo, salvar_pacote)
from equifax.experimentos import (escolher_vencedor, escorar, ler_registro, resumir, rodar_experimento,
                                  treinar)
from equifax.explicar import (conferir_soma, contribuicoes, grafico_beeswarm, grafico_dependencia,
                              grafico_importancia, importancia_shap)
from equifax.features import Cep, ajustar_familias, regiao_do_estado
from equifax.macro import JuntarMacro, montar_tabela_macro
from equifax.tratamento import ORDEM_RENDAS
from equifax.validacao import avaliar
from equifax.viz import salvar_figura

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

In [ ]:
PARAMS_M1 = json.load(open(REPORTS / "params_m1.json", encoding="utf-8"))
M2 = json.load(open(REPORTS / "variaveis_m2.json", encoding="utf-8"))
M3 = json.load(open(REPORTS / "variaveis_m3.json", encoding="utf-8"))
CONJUNTO = PARAMS_M1["conjunto"]
PARAMS_LGBM = PARAMS_M1["lgbm"]
assert M2["base"] == M3["base"] == CONJUNTO
assert set(M2["familias"]) <= {"f4_cep"}, "este notebook só sabe aplicar a família de CEP"

MODELOS = {
    "M1": {"variaveis": carregar_conjunto(CONJUNTO), "cep": False, "macro": []},
    "M2": {"variaveis": M2["variaveis"], "cep": "f4_cep" in M2["familias"], "macro": []},
    "M3": {"variaveis": M3["variaveis"], "cep": "f4_cep" in M2["familias"], "macro": M3["series_macro"]},
}
for nome, m in MODELOS.items():
    print(f"{nome}: {len(m['variaveis'])} variáveis; CEP: {m['cep']}; macro: {m['macro']}")

base = carregar_desenvolvimento()
tabela_macro = montar_tabela_macro()

In [ ]:
def preparar(nome, treino, outras):
    """Ajusta no treino as transformações do modelo (CEP, macro) e aplica nas outras bases.

    Devolve (treino, outras, transformações); as transformações vão no pacote da escoragem.
    """
    m = MODELOS[nome]
    transformacoes = []
    if m["cep"]:
        treino, aplicar_cep = ajustar_familias(treino, [Cep(suavizacao=200)])
        transformacoes.append(aplicar_cep)
    if m["macro"]:
        juntar = JuntarMacro(tabela_macro, m["macro"])
        treino = juntar(treino)
        transformacoes.append(juntar)

    def aplicar(b):
        for f in transformacoes:
            b = f(b)
        return b

    return treino, {k: aplicar(b) for k, b in outras.items()}, transformacoes

## 7.2 Os três modelos lado a lado (sem a OOT interna)

Tudo vem do registro de experimentos: teste por cliente (202301–202308, notebooks 04 e 06; o M1 é o `c_base_m1` do 04, o mesmo modelo do 03b) e validação temporal (treino 202301–202306, avaliação 202307–202308, notebook 06). Médias de 3 sementes.

In [ ]:
registro = ler_registro()
versao_m3 = M3["versao"].removeprefix("m3_t_")
linhas = {
    "M1": {"teste": ("c_base_m1", f"{CONJUNTO}+base_m1"), "temporal": "m3_t_m1"},
    "M2": {"teste": ("c_m2_combinado", f"{CONJUNTO}+m2"), "temporal": "m3_t_m2"},
    "M3": {"teste": (f"m3_{versao_m3}", f"{CONJUNTO}+{versao_m3}"), "temporal": M3["versao"]},
}

def media_e_desvio(nome, conjunto=None):
    r = registro[registro["nome"] == nome]
    if conjunto is not None:
        r = r[r["conjunto"] == conjunto]
    r = r.drop_duplicates(subset=["nome", "semente"], keep="last")  # só a última rodada
    if r.empty:
        raise ValueError(f"{nome} ({conjunto}) não está no registro: rode o 04 e o 06 antes")
    colunas = ["auc", "ks", "ks_min_safra"]
    return pd.concat([r[colunas].mean(), r[colunas].std().add_suffix("_dp")])

comparacao = pd.DataFrame({
    nome: {"variaveis": len(MODELOS[nome]["variaveis"]),
           "fonte_externa": "BCB/IBGE" if MODELOS[nome]["macro"] else "não",
           **media_e_desvio(*l["teste"]).add_suffix("_teste").to_dict(),
           **media_e_desvio(l["temporal"]).add_suffix("_temporal").to_dict()}
    for nome, l in linhas.items()}).T
comparacao.to_csv(TABLES / "final_comparacao.csv")
comparacao


**Critério de escolha (D7, definido antes de olhar a OOT interna):** o desempenho decide quando a diferença supera o ruído; a estabilidade é medida pelo pior KS entre as safras; a explicabilidade só desempata. Na prática (`escolher_vencedor`): os modelos são percorridos do mais simples para o mais complexo, e um modelo toma o lugar do atual só se ganhar de AUC acima do desvio entre sementes **no teste por cliente e na validação temporal**, sem que o pior KS entre safras caia além do ruído. Entre empatados, fica o mais simples: menos variáveis, sem fonte externa e menos coisa para manter em produção. A logística com WoE fica como a alternativa explicável (scorecard), com o custo em perdas medido no `03`.


In [ ]:
VENCEDOR, motivo = escolher_vencedor(comparacao, ordem=("M1", "M2", "M3"))
VARIAVEIS = MODELOS[VENCEDOR]["variaveis"]
print("Vencedor:", VENCEDOR, "|", len(VARIAVEIS), "variáveis")
motivo.round(4)


## 7.3 Confirmação única na OOT interna

Cada modelo é treinado com todas as linhas de 202301–202308 (treino e teste juntos) e avaliado em 202309–202310, os dois meses mais recentes, como se fossem o futuro. A escolha já foi feita: esta tabela confirma, não decide.

In [ ]:
partes = janelas(base, incluir_oot_interna=True)
dev8 = pd.concat([partes["treino"], partes["teste"]])
oot_interna = partes["oot_interna"]
print(f"treino 202301-202308: {len(dev8)} linhas | OOT interna: {len(oot_interna)} linhas, "
      f"inadimplência {1 - oot_interna['adimplencia'].mean():.4f}")

for nome in MODELOS:
    treino_m, outras, _ = preparar(nome, dev8, {"oot_interna": oot_interna})
    rodar_experimento(f"final_{nome.lower()}_oot_interna", MODELOS[nome]["variaveis"],
                      {"treino": treino_m, **outras}, modelo="lgbm", params=PARAMS_LGBM,
                      avaliar_em="oot_interna", usar_oot_interna=True, nome_conjunto=f"{CONJUNTO}:{nome}")
    print(nome, "pronto")

registro = ler_registro()
registro = registro[registro["conjunto"].str.startswith(f"{CONJUNTO}:")]  # só o conjunto atual
confirmacao = resumir(registro, nomes=[f"final_{n.lower()}_oot_interna" for n in MODELOS])
confirmacao.to_csv(TABLES / "final_oot_interna.csv")
confirmacao

## 7.4 O vencedor na OOT interna: o que muda para o negócio

Modelo vencedor treinado em 202301–202308 (semente 42). KS e AUC em cada mês (KS1 = 202309, KS2 = 202310), decis, calibração e a simulação de política: aprovar os clientes de maior score até uma taxa de aprovação.

In [ ]:
treino_v, outras_v, _ = preparar(VENCEDOR, dev8, {"oot_interna": oot_interna})
modelo_v = treinar("lgbm", PARAMS_LGBM, treino_v, VARIAVEIS, semente=RANDOM_SEED)
oot_v = outras_v["oot_interna"]
score_oot = escorar(modelo_v, oot_v)
y_oot, safra_oot = oot_v["adimplencia"], oot_v["safra"]

avaliar(y_oot, score_oot, safra_oot).round(4)

In [ ]:
decis = tabela_decis(y_oot, score_oot)
decis.to_csv(TABLES / "final_decis_oot_interna.csv")
decis.round(4)

In [ ]:
politica = simular_politica(y_oot, score_oot)
politica.to_csv(TABLES / "final_politica_oot_interna.csv")
tabela_cal, brier = tabela_calibracao(y_oot, score_oot)
print(f"Inadimplência média da OOT interna: {1 - y_oot.mean():.2%} | Brier {brier:.4f}")
grafico_calibracao(y_oot, score_oot, f"Calibração na OOT interna ({VENCEDOR})")
salvar_figura("final_calibracao_oot_interna")
plt.show()
politica.round(4)

## 7.5 O vencedor funciona igual para todos os perfis?

AUC e KS do vencedor na OOT interna dentro de cada faixa de renda, faixa de idade e região. Grupos com menos de 1.000 clientes ficam marcados: as métricas deles variam muito por acaso.

In [ ]:
segmentos = []
grupos = {"faixa_renda": oot_v["faixa_renda"], "faixa_idade": oot_v["faixa_idade"],
          "regiao": regiao_do_estado(oot_v["estado"])}  # a região só agrupa os estados para ler o resultado
for coluna, grupo in grupos.items():
    m = avaliar(y_oot, score_oot, grupo.astype(str)).iloc[1:]
    if coluna == "faixa_renda":  # da menor para a maior renda
        m = m.set_index("safra").loc[[r for r in ORDEM_RENDAS if r in set(m["safra"])]].reset_index()
    segmentos.append(m.rename(columns={"safra": "grupo"}).assign(segmento=coluna))
segmentos = pd.concat(segmentos, ignore_index=True)[["segmento", "grupo", "n", "taxa_inad", "auc", "ks"]]
segmentos["poucos_clientes"] = segmentos["n"] < 1000
segmentos.to_csv(TABLES / "final_segmentos_oot_interna.csv", index=False)

fig, eixos = plt.subplots(1, 3, figsize=(15, 5))
for ax, (coluna, g) in zip(eixos, segmentos.groupby("segmento", sort=False)):
    cores = np.where(g["poucos_clientes"], "lightgray", "steelblue")
    ax.barh(g["grupo"], g["ks"], color=cores)
    ax.invert_yaxis()  # primeira categoria em cima
    ax.set_title(coluna)
    ax.set_xlabel("KS na OOT interna")
    ax.grid(axis="x", alpha=.3)
fig.suptitle(f"KS do {VENCEDOR} por perfil (cinza: menos de 1.000 clientes)")
fig.tight_layout()
salvar_figura("final_ks_por_segmento")
plt.show()
segmentos.round(4)

## 7.6 Por que o vencedor decide assim

Contribuições SHAP exatas (TreeSHAP do LightGBM) numa amostra de 10 mil clientes da OOT interna, na escala de log-odds de adimplência: positivo aumenta o score (menos risco). A leitura detalhada das VPs está no `05`; aqui o foco é o vencedor e as variáveis que ele tem a mais que o M1.

In [ ]:
contrib, valor_base, x = contribuicoes(modelo_v, oot_v, amostra=10000, semente=RANDOM_SEED)
ok, erro = conferir_soma(modelo_v, contrib, valor_base, x)
print(f"Soma confere: {ok} (maior erro {erro:.1e})")
imp = importancia_shap(contrib)
imp.to_frame("media_abs_shap").to_csv(TABLES / "final_shap_importancia.csv")

grafico_importancia(contrib, n=20, titulo=f"Importância SHAP na OOT interna ({VENCEDOR})")
salvar_figura("final_shap_importancia")
plt.show()
grafico_beeswarm(contrib, valor_base, x, n=15, titulo=f"Direção e dispersão do efeito ({VENCEDOR})")
salvar_figura("final_shap_beeswarm")
plt.show()

novas = [v for v in VARIAVEIS if v.startswith(("fe_", "macro_"))]
posicao = imp.rank(ascending=False).astype(int)
if novas:
    display(pd.DataFrame({"media_abs_shap": imp[novas], "posicao": posicao[novas]}).sort_values("posicao").round(4))
else:
    print(f"O {VENCEDOR} usa só as colunas fornecidas. As 10 principais variáveis:")
    display(imp.head(10).round(4).to_frame("media_abs_shap"))

In [ ]:
for var in [v for v in ["fe_cep3_taxa_inad", "macro_desocupacao_uf_menos_br"] if v in VARIAVEIS]:
    grafico_dependencia(contrib, x, var)
    salvar_figura(f"final_shap_dependencia_{var}")
    plt.show()

## 7.7 Modelo final: retreino com as 10 safras

Mesmos hiperparâmetros, agora com 202301–202310. As transformações (taxa por prefixo de CEP e, se houver, a junção macro) são ajustadas de novo com as 10 safras e vão no pacote junto com o modelo. Antes de escorar a OOT, o caminho completo (CSV bruto → tratamento → transformações → score) é conferido com 5 mil linhas do CSV de desenvolvimento.

In [ ]:
treino_10, _, transformacoes = preparar(VENCEDOR, base, {})
final = treinar("lgbm", PARAMS_LGBM, treino_10, VARIAVEIS, semente=RANDOM_SEED)
print(f"Treinado com {len(treino_10)} linhas, safras {treino_10['safra'].min()}-{treino_10['safra'].max()}, "
      f"em {final.tempo_treino_s:.0f}s")

pacote = salvar_pacote(final, MODELS / "vencedor.joblib", treino_10, colunas_brutas_do_desenvolvimento(),
                       nome=f"{VENCEDOR}: LightGBM, 10 safras", transformacoes=transformacoes,
                       observacoes="Modelo entregue na OOT cega")
ensaiar_com_desenvolvimento(pacote, base, n=5000)

## 7.8 Escoragem da OOT cega

O arquivo original volta intacto, com a coluna `score` no fim, em `submission/base_oot.csv` (fora do Git). As checagens param a escoragem se colunas, número de linhas, ordem ou ids não baterem; VPs zeradas numa safra, categorias novas e a idade dos dados macro vão para o relatório `submission/base_oot.relatorio.json`.

In [ ]:
saida, relatorio = escorar_arquivo(OOT_PATH, pacote, SUBMISSION / "base_oot.csv", n_linhas_esperado=None,  # número de linhas informado pela organização
                                  nome_coluna="score", tipo_score="probabilidade", separador_saida=";")
print("Linhas:", relatorio["linhas"], "| por safra:", relatorio["safras"])
print("Chaves (id, safra) repetidas:", relatorio["chaves_id_safra_repetidas"])
print("VPs zeradas numa safra:", relatorio["vps_zeradas_por_safra"] or "nenhuma")
print("Categorias novas:", relatorio["categorias_novas"] or "nenhuma")
print("Idade dos dados macro (meses):", relatorio.get("idade_dados_macro_meses", "modelo sem macro"))
print("Score:", {k: round(v, 4) if isinstance(v, float) else v for k, v in relatorio["score"].items()})

## 7.9 A distribuição do score se mantém na OOT?

O regulamento pede a distribuição do score nas safras da OOT. O gráfico mostra o score do modelo final em cada safra do desenvolvimento (em que ele foi treinado) e nas duas safras da OOT cega; o PSI compara cada safra com as duas mais recentes do desenvolvimento (202309–202310). Como o modelo viu o desenvolvimento no treino, esse lado fica um pouco mais espalhado; mesmo assim, PSI abaixo de 0,10 indica estabilidade.

In [ ]:
score_dev = escorar(final, treino_10)
score_oot_cega = saida["score"].astype(float).to_numpy()
todas_safras = np.r_[treino_10["safra"].astype(int).to_numpy(), saida["safra"].astype(int).to_numpy()]
todos_scores = np.r_[score_dev, score_oot_cega]

grafico_distribuicao_score(todos_scores, todas_safras,
                           f"Score do modelo final por safra ({VENCEDOR}; 202311 e 202312 = OOT cega)")
salvar_figura("final_distribuicao_score_oot")
plt.show()
psi = psi_score_por_safra(todos_scores, todas_safras, safras_ref=[202309, 202310])
psi.rename("psi").to_frame().T.round(4)

> Conclusão omitida na versão pública: ela traz números e achados da base, que é confidencial.